## **Training of MODEL2**

In [13]:
import pandas as pd
import numpy as np
from keras.src.ops import dtype
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import joblib
import os

In [14]:
TARGET_VARIABLE = 'Time to Depletion'

In [15]:
data = pd.read_csv("../new_code/DATASET2.csv")

In [16]:
data.head()

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,Time to Depletion,type,capacity,charged,prediction
0,9.36,11.84,0.156000,0.156000,110.8224,26.844000,10324.615385,b2,88.81,27.0,10395.389494
1,9.34,11.84,0.155667,0.311667,110.5856,26.688333,10286.723769,b2,88.81,27.0,10383.381179
2,9.34,11.83,0.155667,0.467333,110.4922,26.532667,10226.723769,b2,88.81,27.0,10391.152743
3,7.14,11.88,0.119000,0.586333,84.8232,26.413667,13317.815126,b2,88.81,27.0,13311.084443
4,7.13,11.88,0.118833,0.705167,84.7044,26.294833,13276.493689,b2,88.81,27.0,13279.109429


In [17]:
Y = data[TARGET_VARIABLE]
X = data.drop(TARGET_VARIABLE, axis=1)

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical Featrures are : ", numerical_features)
print("Categorical Featrures are : ", categorical_features)

Numerical Featrures are :  ['Current', 'Voltage', 'Ah Out', 'Cumulative Actual Disch Ah', 'Power', 'Remaining Capacity', 'capacity', 'charged', 'prediction']
Categorical Featrures are :  ['type']


In [18]:
print("NaN locations:")
for column in data.columns:
    if data[column].isna().any():
        print(f"\n{column}:")
        print(data[data[column].isna()].index)


NaN locations:


In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.15, random_state=42)

In [20]:
numerical_transformer = Pipeline(steps=[
    ('pass',
     'passthrough')
])
categorical_transformer = Pipeline(steps=[
    ('onehot',
     OneHotEncoder(handle_unknown='ignore',
                   sparse_output=False))
])

In [21]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
]
    ,remainder='passthrough')

In [22]:
rf_model = RandomForestRegressor(
    random_state=42,
    bootstrap=True,
    criterion='absolute_error',
    n_jobs=-1,
    n_estimators=100
)

In [23]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', rf_model)
])

In [24]:
param_grid = {
    'regressor__n_estimators' : [100,150,200],
    'regressor__max_depth': [10,12,14],
    'regressor__min_samples_split': [2,3,4],
    'regressor__min_samples_leaf': [1,2,3]
}

In [25]:
grid_search = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring = 'neg_mean_absolute_error',
    cv = 2,
    verbose = 2,
    n_jobs = -1,
    return_train_score = True,
    refit = True
)

In [26]:
print("Initiating the Grid Search...")
grid_search.fit(X_train, Y_train)
print("Search Finished")

Initiating the Grid Search...
Fitting 2 folds for each of 81 candidates, totalling 162 fits
Search Finished


In [27]:
best_match = grid_search.best_estimator_

In [28]:
Y_pred = best_match.predict(X_test)

In [29]:
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error is : {mae:.2f}")

Mean Absolute Error is : 122.25


In [30]:
joblib.dump(best_match, "../models/battery_random_forest_model2.joblib")

['../models/battery_random_forest_model2.joblib']

In [31]:
loaded_model = joblib.load("../models/battery_random_forest_model2.joblib")


In [32]:
y_loaded_pred = loaded_model.predict(X_test)

mae_loaded = mean_absolute_error(Y_test, y_loaded_pred)
print(f"Mean Absolute Error is : {mae_loaded:.2f}")

Mean Absolute Error is : 122.25
